In [ ]:
import jax
jax.config.update("jax_enable_x64", True)


In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

# --- new (scene) API ---------------------------------------------------------
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.cosmo import w0waCDM_Cosmo
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.simulator import SimulatorConfig

# --- research-side pipeline / diagnostics (gigalens_research) ----------------
from gigalens_research.inference_utils import (
    InferenceContext, Pipeline, MAPStage, BridgeStage, MCLMCStage,
)
from gigalens_research.plotting import PosteriorReport, PipelineReport


In [ ]:
z_lens = 0.5
z_source1 = z_lens * 2
z_source2 = z_lens * 3

lens = Component(
    EPL(),
    dict(
        theta_E=tfd.LogNormal(jnp.log(1.25), 0.25),
        gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
        e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
        center_x=tfd.Normal(0, 0.05),
        center_y=tfd.Normal(0, 0.05),
    ),
)
# NOTE: the old notebook also drafted a `shear` Prior/Component (mass.shear.Shear())
# but commented it out of the model it actually built and ran; omitted here too.

source1 = Component(
    SersicEllipse(use_lstsq=False),
    dict(
        center_x=tfd.Normal(0, 2),
        center_y=tfd.Normal(0, 2),
        e1=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        n_sersic=tfd.Uniform(1, 10),
        R_sersic=tfd.LogNormal(jnp.log(1.), 0.15),
        Ie=tfd.LogNormal(jnp.log(150), 1),
    ),
)
source2 = Component(
    SersicEllipse(use_lstsq=False),
    dict(
        center_x=tfd.Normal(0, 2),
        center_y=tfd.Normal(0, 2),
        e1=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        n_sersic=tfd.Uniform(1, 10),
        R_sersic=tfd.LogNormal(jnp.log(1.), 0.15),
        Ie=tfd.LogNormal(jnp.log(150), 1),
    ),
)


def tNCDF_bij(low, high):
    return tfb.Chain([tfb.Shift(low), tfb.Scale(high - low), tfb.NormalCDF()])


# Verbatim from the old notebook: a tfd.Uniform with a Shift/Scale/NormalCDF
# event-space bijector in place of TFP's default (sigmoid-based) one for Uniform.
class UniformBij(tfd.Uniform):
    def __init__(self, *args, event_space_bijector_class=tNCDF_bij, **kwargs):
        self._esb = event_space_bijector_class(*args)
        super().__init__(*args, **kwargs)

    def _default_event_space_bijector(self):
        return self._esb
cosmo = Component(
    w0waCDM_Cosmo(z_lens=z_lens, z_source_ref=z_source1),
    {
        "H0":70.,
        # ("Om0", "w0"): ratio_coords.UFirstRatioCoordsUniform(u_fn, (0.01, 0.99), (-2.0, -1/3), du_dw_atol=1.8e-3, excursion_atol=2e-4, curve_atol=0.0),
        "Om0":0.3,#UniformBij(jnp.float64(0.1), jnp.float64(0.8)),
        "w0":-1.0,#UniformBij(jnp.float64(-2.0), jnp.float64(-1 / 3)),
        "wa":0.0,
        "k":0.0,
    },
)


# "wa": 

model = LensModel(
    [
        Plane(redshift=z_lens, mass=[lens]),
        Plane(redshift=z_source1, light=[source1]),
        Plane(redshift=z_source2, light=[source2]),
    ],
    cosmo=cosmo,
)

print(f"num_free_params = {model.num_free_params}")
print("z_param_names   =", model.z_param_names)


In [ ]:
truth_scene = {
    "planes": {
        0: {
            "geometry": {"redshift": z_lens},
            "mass": {
                0: {"theta_E": 1.1, "gamma": 2.0, "e1": 0.05, "e2": 0.02,
                    "center_x": 0.0, "center_y": 0.0},
            },
        },
        1: {
            "geometry": {"redshift": z_source1},
            "light": {
                0: {"R_sersic": 0.25, "n_sersic": 2., "e1": 0.05, "e2": 0.,
                    "center_x": 0.05, "center_y": 0., "Ie": 50.},
            },
        },
        2: {
            "geometry": {"redshift": z_source2},
            "light": {
                0: {"R_sersic": 1., "n_sersic": 6., "e1": 0.0, "e2": 0.05,
                    "center_x": 0., "center_y": 0.05, "Ie": 15.},
            },
        },
    },
    "cosmo": dict(H0=70.0, Om0=0.3, k=0.0, w0=-1.0, wa=0.0),
}


In [ ]:
numPix, deltaPix, exp_time, background_rms = 120, 0.065/2, 1000, 0.1
extent = (-numPix / 2 * deltaPix, numPix / 2 * deltaPix,
          -numPix / 2 * deltaPix, numPix / 2 * deltaPix)

import photutils.psf as psf
kernel = psf.GaussianPSF(x_fwhm=2, y_fwhm=2)
yy, xx = np.mgrid[-7:8, -7:8]
kernel = kernel(xx, yy)

sim_config = SimulatorConfig(
    delta_pix=deltaPix,
    num_pix=numPix,
    kernel=kernel,
    supersample=1,
    likelihood_precision="float64",
)

sim_config_truth = SimulatorConfig(
    delta_pix=deltaPix,
    num_pix=numPix,
    kernel=kernel,
    supersample=1,
    likelihood_precision="float64",
)


In [ ]:
from lenstronomy.Util import image_util


def add_noise(img, exp_time, sigma_bkd):
    poisson = image_util.add_poisson(img, exp_time=exp_time)
    bkg = image_util.add_background(img, sigma_bkd=sigma_bkd)
    return img + poisson + bkg


sim1 = SceneSimulator(model, sim_config_truth, sees=[source1])
sim2 = SceneSimulator(model, sim_config_truth, sees=[source2])
print("trace mode (both simulators, same single-mass-plane model):",
      sim1.trace_mode, sim2.trace_mode)

img1 = np.asarray(sim1.simulate(truth_scene))
img2 = np.asarray(sim2.simulate(truth_scene))

observed_image1 = add_noise(img1, exp_time=exp_time, sigma_bkd=background_rms)
observed_image2 = add_noise(img2, exp_time=exp_time, sigma_bkd=background_rms)

plt.figure(figsize=(8, 3))
ax = plt.subplot(121)
plt.imshow(observed_image1, extent=extent)
plt.colorbar()
ax = plt.subplot(122)
plt.imshow(observed_image2, extent=extent)
plt.colorbar()
plt.show()


## Probabilistic model

`mode="forward"` matches the old notebook's `ForwardMultiModel` (light
amplitudes, `Ie`, are sampled rather than lstsq-solved — consistent with
`SersicEllipse(use_lstsq=False)` above). `Dataset(..., background_rms=...,
exp_time=...)` computes `error_map = sqrt(background_rms**2 +
clip(image, 0, inf) / exp_time)` — checked against
`gigalens-old/src/gigalens/jax/prob_model.py::ForwardMultiModel.__init__`,
which uses the **identical** formula, so this is not an approximation.

`sees=[source1]` / `sees=[source2]` encode "band 1 only shows source 1, band 2
only shows source 2" — the same per-band separation the old notebook's
`multiband_simulate` produced.


In [ ]:
dataset1 = ImageData(observed_image1, sim_config, background_rms=background_rms,
                    exp_time=exp_time, sees=[source1])
dataset2 = ImageData(observed_image2, sim_config, background_rms=background_rms,
                    exp_time=exp_time, sees=[source2])

prob_model = ProbModel(model, [dataset1, dataset2], mode="forward")


# ctx = InferenceContext.from_prob_model(prob_model)

# pipeline = Pipeline(ctx, seed=42)


In [ ]:
from gigalens.jax.lens_solver import LensSolver,plot_image_positions

In [ ]:
ls = LensSolver(prob_model)

In [ ]:
ls.solve_scene(truth_scene)

In [ ]:
extent2 = (-numPix / 2 * deltaPix, numPix / 2 * deltaPix,
          numPix / 2 * deltaPix, -numPix / 2 * deltaPix)
plt.figure(figsize=(8, 3))
ax = plt.subplot(121)
plt.imshow(observed_image1, extent=extent2, origin="lower")
plot_image_positions(ax, ls, truth_scene, planes=[1])
plt.colorbar()
ax = plt.subplot(122)
plt.imshow(observed_image2, extent=extent2, origin="lower")
plot_image_positions(ax, ls, truth_scene, planes=[2])
plt.colorbar()
plt.show()
